# Notebook 2 — Preprocessing & Tokenization
Formats dataset into TinyLlama instruction template and splits into train/val/test.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
os.chdir('/content/drive/MyDrive/medical_qa_project')

In [3]:
!pip install -q datasets transformers

In [ ]:
from datasets import load_from_disk
from transformers import AutoTokenizer
import os

# ── Load saved dataset ──
dataset = load_from_disk("dataset/medical_qa_raw")
print(f"Loaded {len(dataset)} examples")

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_14519/1486642103.py", line 1, in <cell line: 0>
    from datasets import load_from_disk
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1322, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 1262, in _find_spec
  File "<frozen importlib._bootstrap_external>", line 1532, in find_spec
  File "<frozen importlib._bootstrap_external>", line 1504, in _get_spec
  File "<frozen importlib._bootstrap_external>", line 1483, in _path_importer_cache
OSError: [Errno 107] Transport endpoint is not connected

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 20

In [ ]:
# ── Format into TinyLlama instruction template ──
def format_example(example):
    text = f"""### Instruction:
{example['instruction']}
### Response:
{example['output']}"""
    return {"text": text}

dataset = dataset.map(format_example)
print("Sample formatted text:")
print(dataset[0]["text"][:400])

In [ ]:
# ── Analyze token lengths ──
import matplotlib.pyplot as plt

lengths = [len(tokenizer(x["text"])["input_ids"]) for x in dataset]
print(f"Min tokens: {min(lengths)}")
print(f"Max tokens: {max(lengths)}")
print(f"Mean tokens: {sum(lengths)/len(lengths):.1f}")

plt.figure(figsize=(8,4))
plt.hist(lengths, bins=40, color="steelblue", edgecolor="white")
plt.axvline(256, color="red", linestyle="--", label="max_length=256")
plt.xlabel("Token Length")
plt.ylabel("Count")
plt.title("Token Length Distribution")
plt.legend()
plt.tight_layout()
plt.savefig("dataset/token_distribution.png", dpi=120)
plt.show()
print(f"Examples ≤ 256 tokens: {sum(1 for l in lengths if l<=256)} / {len(lengths)}")

In [ ]:
# ── Tokenize ──
def tokenize(example):
    tok = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )
    tok["labels"] = tok["input_ids"].copy()
    return tok

tokenized = dataset.map(tokenize, batched=True, remove_columns=dataset.column_names)
print("Tokenized columns:", tokenized.column_names)

In [ ]:
# ── Train / Val / Test split (80 / 10 / 10) ──
split1 = tokenized.train_test_split(test_size=0.2, seed=42)
split2 = split1["test"].train_test_split(test_size=0.5, seed=42)

train_ds = split1["train"]
val_ds   = split2["train"]
test_ds  = split2["test"]

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

os.makedirs("dataset", exist_ok=True)
train_ds.save_to_disk("dataset/train")
val_ds.save_to_disk("dataset/val")
test_ds.save_to_disk("dataset/test")
print("Splits saved.")